# 04 — Otimização de Hiperparâmetros Rigorosa (Optuna) no Fashion-MNIST

## Objetivo Científico
No repositório `mnist-study`, a busca por hiperparâmetros com Optuna (TPE + MedianPruner) **não** superou
a CNN baseline de forma estatisticamente significativa (Δ = +0,01pp, t pareado p = 0,885). A hipótese explicativa
foi o **efeito de teto** (*ceiling effect*) do MNIST tradicional (~99% de acurácia com modelo simples).

O **Fashion-MNIST** possui um teto de acurácia consideravelmente mais baixo (~92–93%), oferecendo maior margem
para ganhos reais com tuning. O objetivo deste notebook é executar **exatamente a mesma metodologia rigorosa**
usada no MNIST para testar se a otimização de hiperparâmetros gera um ganho significativo no Fashion-MNIST —
o que confirmaria que o resultado nulo do MNIST foi efeito do dataset, e não falha do método.

## Metodologia Rigorosa
1. **Busca (Optuna TPE + MedianPruner):**
   - Subset representativo de **20.000 amostras** do split de treino (50k);
   - **10 épocas por trial**, 40 trials no total;
   - A **CNN baseline é enfileirada no 1º trial** (`study.enqueue_trial`), garantindo que o melhor trial seja ≥ baseline por construção;
   - Estudo persistido em `results/optuna_study_fashion.db` (resiliente a interrupções).

2. **Avaliação Multi-Seed Pareada (5 seeds):**
   - Retreino do zero da **baseline** e da **melhor configuração** no split completo (50k treino, 10 épocas, 5 seeds pareadas);
   - Restauração automática dos pesos da melhor época na validação (*restore best val*);
   - **Teste t pareado** (`stats.ttest_rel`), **Cohen's d_z pareado**, **IC 95%** da diferença média e **Teste de McNemar**.

3. **Seleção Final na Validação:**
   - A decisão de adotar o modelo tunado é feita estritamente via `best_val_mean >= baseline_val_mean`, evitando vazamento (*data leakage*) ou viés de seleção no conjunto de teste.

In [ ]:
# ── Setup — funciona igual no seu PC e no Google Colab ───────────────────────
import sys, os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/4rth-g/fashion-mnist-fundamentos-ia.git"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not Path("fashion-mnist-fundamentos-ia").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("fashion-mnist-fundamentos-ia")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna", "seaborn"], check=True)

# Torna o pacote src/ importável (local: rodando de notebooks/; Colab: da raiz).
_root = Path.cwd()
_src = _root / "src" if (_root / "src").exists() else _root.parent / "src"
sys.path.insert(0, str(_src))
print("src/ em:", _src)

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import optuna
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from scipy import stats
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from torch.utils.data import DataLoader, Subset

from utils import (
    CLASS_NAMES,
    FIGURES_DIR,
    MODELS_DIR,
    RESULTS_DIR,
    SEED,
    get_device,
    loader_kwargs,
    make_dataloaders,
    mcnemar_test,
    log_experiment,
    seed_everything,
)

optuna.logging.set_verbosity(optuna.logging.WARNING)

seed_everything()
device = get_device()
kw = loader_kwargs(device)

## 1. Carregamento e Divisão dos Dados

Carregamos os DataLoaders oficiais do repositório (`50k treino`, `10k validação`, `10k teste`).
Data augmentation leve (`RandomCrop` + `RandomHorizontalFlip`) é aplicada apenas no treino.

In [ ]:
train_loader, val_loader, test_loader = make_dataloaders(batch_size=128, device=device, augment=True)
print(f"Treino (split) : {len(train_loader.dataset)} amostras")
print(f"Validação      : {len(val_loader.dataset)} amostras")
print(f"Teste          : {len(test_loader.dataset)} amostras  ← usado apenas na avaliação final")

## 2. Arquitetura CNN Configurável com BatchNorm

Para permitir que a otimização explore tanto a capacidade quanto a regularização,
usamos a arquitetura com **Batch Normalization** em cada bloco convolucional, idêntica ao template de referência.

In [ ]:
class CNN(nn.Module):
    """
    CNN configurável com BatchNorm para busca de hiperparâmetros.

    Arquitetura:
        Conv1 -> BN -> ReLU -> MaxPool (28x28 -> 14x14)
        Conv2 -> BN -> ReLU -> MaxPool (14x14 -> 7x7)
        Flatten -> FC1 -> ReLU -> Dropout -> FC2 (10 classes)
    """

    def __init__(self, num_filters=32, dropout_rate=0.5, fc_units=128, n_conv_blocks=2):
        super().__init__()
        layers = []
        in_ch = 1
        for b in range(n_conv_blocks):
            out_ch = num_filters * (2**b)
            layers += [
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(),
                nn.MaxPool2d(2, 2),
            ]
            in_ch = out_ch
        self.features = nn.Sequential(*layers)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(fc_units),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(fc_units, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


def make_optimizer(model, config):
    lr = config["lr"]
    wd = config.get("weight_decay", 0.0)
    if config["optimizer"] == "adam":
        return optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    if config["optimizer"] == "adamw":
        return optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    return optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)


# Sanity check da arquitetura
model_test = CNN().to(device)
dummy = torch.randn(4, 1, 28, 28, device=device)
out = model_test(dummy)
print(f"Output shape: {out.shape} (esperado: [4, 10])")
print(f"Parâmetros totais (config padrão): {sum(p.numel() for p in model_test.parameters()):,}")
del model_test, dummy

## 3. Helpers Locais de Treino e Avaliação

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct = 0.0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (output.argmax(1) == labels).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n


def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0.0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            output = model(images)
            total_loss += criterion(output, labels).item() * images.size(0)
            preds = output.argmax(1)
            correct += (preds == labels).sum().item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    n = len(loader.dataset)
    return total_loss / n, correct / n, np.array(all_preds), np.array(all_labels)

## 4. Otimização de Hiperparâmetros (Optuna TPE + MedianPruner)

Espaço de busca:
- `num_filters`: `[16, 32, 64, 128]`
- `dropout_rate`: `[0.0, 0.6]`
- `fc_units`: `[64, 128, 256, 512]`
- `lr`: `[1e-4, 5e-3]` (escala logarítmica)
- `optimizer`: `["adam", "adamw", "sgd"]`
- `weight_decay`: `[1e-6, 1e-2]` (escala logarítmica)
- `batch_size`: `[64, 128]`

A **baseline** (`num_filters=32, dropout=0.5, fc=128, lr=1e-3, bs=128, optimizer=adam`) é enfileirada no 1º trial.

In [ ]:
GS_SUBSET_SIZE = 20_000
EPOCHS_GS = 10
N_TRIALS = 40

gs_indices = torch.randperm(
    len(train_loader.dataset), generator=torch.Generator().manual_seed(SEED)
)[:GS_SUBSET_SIZE]
gs_dataset = Subset(train_loader.dataset, gs_indices.tolist())
print(f"Subset para busca: {len(gs_dataset)} amostras (de {len(train_loader.dataset)} do split de treino)")

baseline_config = {
    "num_filters": 32,
    "dropout_rate": 0.5,
    "fc_units": 128,
    "lr": 1e-3,
    "batch_size": 128,
    "optimizer": "adam",
    "weight_decay": 0.0,
    "n_conv_blocks": 2,
}
print("Config baseline:", baseline_config)


def objective(trial):
    config = {
        "num_filters": trial.suggest_categorical("num_filters", [16, 32, 64, 128]),
        "dropout_rate": trial.suggest_float("dropout_rate", 0.0, 0.6),
        "fc_units": trial.suggest_categorical("fc_units", [64, 128, 256, 512]),
        "lr": trial.suggest_float("lr", 1e-4, 5e-3, log=True),
        "optimizer": trial.suggest_categorical("optimizer", ["adam", "adamw", "sgd"]),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [64, 128]),
        "n_conv_blocks": trial.suggest_categorical("n_conv_blocks", [2, 3]),
    }

    seed_everything(SEED)  # semente idêntica entre trials para isolar o efeito dos HPs
    gen = torch.Generator().manual_seed(SEED)
    train_dl = DataLoader(
        gs_dataset, batch_size=config["batch_size"], shuffle=True, generator=gen, **kw
    )

    model = CNN(config["num_filters"], config["dropout_rate"], config["fc_units"], config["n_conv_blocks"]).to(device)
    optimizer = make_optimizer(model, config)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    for epoch in range(1, EPOCHS_GS + 1):
        train_epoch(model, train_dl, optimizer, criterion, device)
        _, val_acc, _, _ = eval_epoch(model, val_loader, criterion, device)
        best_val_acc = max(best_val_acc, val_acc)

        trial.report(val_acc, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_val_acc


STUDY_DB = RESULTS_DIR / "optuna_study_fashion_depth.db"
study = optuna.create_study(
    study_name="cnn_fashion_gridsearch_depth",
    storage=f"sqlite:///{STUDY_DB}",
    load_if_exists=True,
    direction="maximize",
    sampler=TPESampler(seed=SEED),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)

if len(study.trials) == 0:
    baseline_trial = dict(baseline_config)
    baseline_trial["weight_decay"] = max(baseline_trial["weight_decay"], 1e-6)
    study.enqueue_trial(baseline_trial)

remaining = max(0, N_TRIALS - len(study.trials))
print(f"Trials já no estudo: {len(study.trials)} | faltam: {remaining}")
if remaining > 0:
    study.optimize(objective, n_trials=remaining, show_progress_bar=True)

print(f"Busca concluída — {len(study.trials)} trials no total")

## 5. Análise dos Resultados da Busca & Visualização do Histórico

In [ ]:
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
print(f"Completos: {len(completed)} | Podados: {len(pruned)}")
print(f"\nMelhor val_acc na busca: {study.best_value:.4f} ({study.best_value * 100:.2f}%)")
print("Melhor configuração encontrada:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

df_trials = study.trials_dataframe()
trials_csv_path = RESULTS_DIR / "optuna_trials_fashion_depth.csv"
df_trials.to_csv(trials_csv_path, index=False)
print(f"Tabela de trials salva em {trials_csv_path}")

from optuna.visualization.matplotlib import plot_optimization_history

fig = plot_optimization_history(study)
fig.figure.set_size_inches(10, 5)
fig.set_title("Histórico da Otimização Optuna — Fashion-MNIST")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fashion_depth_optuna_history.png", dpi=150)
plt.show()

## 6. Retreinamento Multi-Seed Pareado (5 seeds)

In [ ]:
def train_full_run(config, seed, epochs=10, verbose=False):
    """Treina no split completo de 50k com restauração dos pesos da melhor época na validação."""
    seed_everything(seed)
    gen = torch.Generator().manual_seed(seed)
    train_dl = DataLoader(
        train_loader.dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        generator=gen,
        **kw,
    )

    model = CNN(config["num_filters"], config["dropout_rate"], config["fc_units"], config["n_conv_blocks"]).to(device)
    optimizer = make_optimizer(model, config)
    criterion = nn.CrossEntropyLoss()

    best_val_acc, best_state, best_epoch = -1.0, None, 0
    for epoch in range(1, epochs + 1):
        train_epoch(model, train_dl, optimizer, criterion, device)
        _, val_acc, _, _ = eval_epoch(model, val_loader, criterion, device)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        if verbose:
            print(f"  seed={seed} epoch={epoch}/{epochs} val_acc={val_acc:.4f}")

    if best_state is not None and best_epoch != epochs:
        model.load_state_dict(best_state)

    _, test_acc, preds, true_labels = eval_epoch(model, test_loader, criterion, device)
    return test_acc, best_val_acc, model, preds, true_labels


N_SEEDS = 5
FINAL_EPOCHS = 10
best_config = dict(study.best_params)
seeds = [SEED + i for i in range(N_SEEDS)]

print(f"Retreinando baseline e melhor config com {N_SEEDS} seeds cada ({FINAL_EPOCHS} épocas)... split 50k")

baseline_runs, baseline_models = [], []
for s in seeds:
    test_acc, val_acc, model, preds, true_labels = train_full_run(baseline_config, s, FINAL_EPOCHS)
    baseline_runs.append({"seed": s, "test_acc": test_acc, "val_acc": val_acc})
    baseline_models.append((model, preds, true_labels, test_acc))
    print(f"  baseline  seed={s}: val_acc={val_acc:.4f} test_acc={test_acc:.4f}")

best_runs, best_models = [], []
for s in seeds:
    test_acc, val_acc, model, preds, true_labels = train_full_run(best_config, s, FINAL_EPOCHS)
    best_runs.append({"seed": s, "test_acc": test_acc, "val_acc": val_acc})
    best_models.append((model, preds, true_labels, test_acc))
    print(f"  melhor    seed={s}: val_acc={val_acc:.4f} test_acc={test_acc:.4f}")

## 7. Comparação Estatística Rigorosa (t Pareado, Cohen's d_z, IC 95% e McNemar)

Calculamos a estatística pareada entre as 5 seeds da baseline e as 5 seeds do modelo otimizado,
além de executar o teste de hipótese não-paramétrico de McNemar entre os melhores modelos de cada lado na validação.

In [ ]:
baseline_acc = np.array([r["test_acc"] for r in baseline_runs])
best_acc = np.array([r["test_acc"] for r in best_runs])

baseline_val_acc_arr = np.array([r["val_acc"] for r in baseline_runs])
best_val_acc_arr = np.array([r["val_acc"] for r in best_runs])
baseline_val_mean = float(baseline_val_acc_arr.mean())
best_val_mean = float(best_val_acc_arr.mean())

diff_acc = best_acc - baseline_acc
t_stat, p_value = stats.ttest_rel(best_acc, baseline_acc)
diff_std = float(diff_acc.std(ddof=1))
cohens_d = float(diff_acc.mean() / diff_std) if diff_std > 0 else 0.0

ci95_tuple = stats.t.interval(0.95, df=len(diff_acc) - 1, loc=diff_acc.mean(), scale=stats.sem(diff_acc))
ci95 = [float(ci95_tuple[0]), float(ci95_tuple[1])]

best_baseline_idx = int(np.argmax(baseline_val_acc_arr))
best_tuned_idx = int(np.argmax(best_val_acc_arr))
preds_baseline_best_val = baseline_models[best_baseline_idx][1]
preds_tuned_best_val = best_models[best_tuned_idx][1]
y_test_labels = best_models[best_tuned_idx][2]

mcnemar_res = mcnemar_test(y_test_labels, preds_baseline_best_val, preds_tuned_best_val)
tuned_ge_baseline = bool(best_val_mean >= baseline_val_mean)

if tuned_ge_baseline:
    selected_name, selected_config = "melhor config", best_config
else:
    selected_name, selected_config = "baseline (fallback)", baseline_config

print("=== Comparação — 5 seeds por config (Fashion-MNIST) ===")
print(f"Baseline (val)     : {baseline_val_mean * 100:.3f}% ± {baseline_val_acc_arr.std(ddof=1) * 100:.3f}%")
print(f"Melhor config (val): {best_val_mean * 100:.3f}% ± {best_val_acc_arr.std(ddof=1) * 100:.3f}%")
print(f"Baseline (teste)   : {baseline_acc.mean() * 100:.3f}% ± {baseline_acc.std(ddof=1) * 100:.3f}%")
print(f"Melhor config(test): {best_acc.mean() * 100:.3f}% ± {best_acc.std(ddof=1) * 100:.3f}%")
print(f"\nPaired t-test (seeds pareadas): t={t_stat:.3f}, p={p_value:.4f}")
print(f"Cohen's d_z: {cohens_d:.3f}")
print(f"IC 95% da diferença média (teste): [{ci95[0]*100:.3f}%, {ci95[1]*100:.3f}%]")
print(f"McNemar test (melhor seed val): n01={mcnemar_res['n01']}, n10={mcnemar_res['n10']}, stat={mcnemar_res['statistic']:.4f}, p={mcnemar_res['p_value']:.4e}, method={mcnemar_res['method']}")
print(f"\n→ Modelo selecionado (via validação): {selected_name}")

gridsearch_json_path = RESULTS_DIR / "gridsearch_rigoroso_fashion_depth.json"
comparison_result = {
    "baseline_config": baseline_config,
    "best_config": best_config,
    "baseline_val_mean": baseline_val_mean,
    "baseline_val_std": float(baseline_val_acc_arr.std(ddof=1)),
    "best_val_mean": best_val_mean,
    "best_val_std": float(best_val_acc_arr.std(ddof=1)),
    "baseline_val_acc": baseline_val_acc_arr.tolist(),
    "best_val_acc": best_val_acc_arr.tolist(),
    "baseline_test_acc": baseline_acc.tolist(),
    "best_test_acc": best_acc.tolist(),
    "baseline_mean": float(baseline_acc.mean()),
    "baseline_std": float(baseline_acc.std(ddof=1)),
    "best_mean": float(best_acc.mean()),
    "best_std": float(best_acc.std(ddof=1)),
    "t_stat": float(t_stat),
    "p_value": float(p_value),
    "cohens_d": cohens_d,
    "ci95": ci95,
    "mcnemar": mcnemar_res,
    "tuned_ge_baseline": tuned_ge_baseline,
    "selected_name": selected_name,
    "selected_config": selected_config,
    "search_best_val_acc": float(study.best_value),
}

with open(gridsearch_json_path, "w") as f:
    json.dump(comparison_result, f, indent=2)
print(f"Resultados completos salvos em {gridsearch_json_path}")

# Boxplot pareado
fig, ax = plt.subplots(figsize=(7, 5))
ax.boxplot(
    [baseline_acc * 100, best_acc * 100],
    tick_labels=["Baseline", "Melhor config (Optuna)"],
)
for i, arr in enumerate([baseline_acc, best_acc], start=1):
    ax.scatter(
        [i] * len(arr),
        arr * 100,
        color="red",
        zorder=3,
        alpha=0.7,
        label="seeds" if i == 1 else None,
    )
ax.set_ylabel("Acurácia no teste (%)")
ax.set_title(f"Baseline vs Optuna — Fashion-MNIST ({N_SEEDS} seeds pareadas, t p={p_value:.3f})")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fashion_depth_comparacao_baseline_gridsearch.png", dpi=150)
plt.show()

## 8. Seleção Final na Validação & Avaliação do Modelo Escolhido no Teste

In [ ]:
use_best_config = best_val_mean >= baseline_val_mean

if use_best_config:
    idx = best_tuned_idx
    final_model, preds, true_labels, final_test_acc = best_models[idx]
    final_config = best_config
    print(f"Modelo final escolhido: melhor config do Optuna (seed={seeds[idx]}, test_acc={final_test_acc:.4f})")
else:
    idx = best_baseline_idx
    final_model, preds, true_labels, final_test_acc = baseline_models[idx]
    final_config = baseline_config
    print(f"Modelo final escolhido: baseline (fallback) (seed={seeds[idx]}, test_acc={final_test_acc:.4f})")

print(f"\nAcurácia final no teste: {final_test_acc * 100:.2f}%")
print("\nRelatório de Classificação por Classe:")
print(classification_report(true_labels, preds, target_names=CLASS_NAMES))

# Matriz de Confusão
cm = confusion_matrix(true_labels, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(8, 7))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Matriz de Confusão — Modelo Final (Fashion-MNIST)", fontsize=13)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fashion_depth_matriz_confusao.png", dpi=150)
plt.show()

# Salvar Checkpoint Final
best_model_path = MODELS_DIR / "fashion_cnn_best_depth.pth"
torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "config": final_config,
        "test_acc": final_test_acc,
        "comparison": comparison_result,
    },
    best_model_path,
)
print(f"Modelo salvo em {best_model_path}")

# Registro no tracker SQLite (results/experiment_tracker.db)
log_experiment(
    model_name="cnn_fashion_baseline",
    config=baseline_config,
    metrics={"val_acc": baseline_val_mean, "test_acc": comparison_result["baseline_mean"], "test_std": comparison_result["baseline_std"]},
)
log_experiment(
    model_name="cnn_fashion_tuned",
    config=final_config,
    metrics={"val_acc": best_val_mean, "test_acc": comparison_result["best_mean"], "test_std": comparison_result["best_std"], "p_value": comparison_result["p_value"], "cohens_d": comparison_result["cohens_d"]},
)
print("Experimentos (baseline + tunado) registrados em results/experiment_tracker.db")